# Capítulo 8 — Resource Management (FreeRTOS)

**Libro:** Mastering the FreeRTOS Real Time Kernel v1.1.0 — Richard Barry  
**Hardware de referencia:** STM32 Nucleo F103RB  
**APIs clave:** `semphr.h`, `task.h`


## 8.1 El Problema del Acceso Compartido a Recursos

En un sistema multitarea, si una tarea comienza a acceder a un recurso compartido pero es desalojada antes de terminar, el recurso puede quedar en un estado inconsistente. Esto puede causar corrupción de datos.

**Ejemplos de problemas:**

**1. Escritura en una LCD:**
1. Tarea A empieza a escribir `"Hello world"` en la LCD.
2. Tarea A es desalojada tras escribir `"Hello w"`.
3. Tarea B escribe `"Abort, Retry, Fail?"` en la LCD.
4. Tarea A retoma y completa `"orld"`.

La LCD muestra: `Hello wAbort, Retry, Fail?orld` ← corrupto.

**2. Operaciones Read-Modify-Write (no atómicas):**

```c
/* En C: */
PORTA |= 0x01;

/* En ensamblador (3 instrucciones no atómicas): */
LOAD  R1, [#PORTA]   ; 1. Lee PORTA
OR    R1, R2          ; 2. Modifica en el registro
STORE R1, [#PORTA]   ; 3. Escribe de vuelta
```

Si otra tarea modifica PORTA entre las instrucciones 1 y 3, se sobrescribe su cambio.

**3. Variables no atómicas y reentrancia:**

Una función es **reentrant** si puede llamarse simultáneamente desde múltiples tareas sin riesgo de corrupción. Solo depende del stack y registros locales.

```c
/* REENTRANT: solo usa variables locales (stack) */
long lAddOneHundred(long lVar1) {
    long lVar2 = lVar1 + 100;
    return lVar2;
}

/* NO REENTRANT: usa variable global compartida */
long lVar1;                    /* global — compartida entre todas las tareas */
long lNonsenseFunction(void) {
    static long lState = 0;    /* static — una sola copia, no thread-safe */
    long lReturn;
    switch (lState) {
        case 0: lReturn = lVar1 + 10; lState = 1; break;
        case 1: lReturn = lVar1 + 20; lState = 0; break;
    }
    return lReturn;
}
```

**Solución general: Exclusión Mutua (Mutual Exclusion)**  
Garantizar que, una vez que una tarea comienza a acceder a un recurso compartido no reentrante, tiene acceso exclusivo hasta dejarlo en estado consistente.


## 8.2 Secciones Críticas

### 8.2.1 Secciones Críticas Básicas

Las secciones críticas rodean código con llamadas a `taskENTER_CRITICAL()` y `taskEXIT_CRITICAL()`. Funcionan **deshabilitando interrupciones** (hasta `configMAX_SYSCALL_INTERRUPT_PRIORITY`), evitando cambios de contexto dentro de ellas.

```c
/* Ejemplo: proteger acceso a un registro de periférico */
taskENTER_CRITICAL();
{
    /* Entre estas dos llamadas NO puede ocurrir un context switch.
       Las interrupciones con prioridad <= configMAX_SYSCALL_INTERRUPT_PRIORITY
       también están deshabilitadas. */
    PORTA |= 0x01;
}
taskEXIT_CRITICAL();
```

**Reglas importantes:**
- Las secciones críticas deben ser MUY cortas. Si son largas, afectan el tiempo de respuesta a interrupciones.
- Se pueden anidar: el kernel lleva la cuenta. La sección solo se sale cuando el contador llega a cero.
- `taskENTER_CRITICAL()` y `taskEXIT_CRITICAL()` NO pueden usarse desde una ISR. Usar `taskENTER_CRITICAL_FROM_ISR()` / `taskEXIT_CRITICAL_FROM_ISR()` en su lugar.

```c
/* Versión para ISR (solo en FreeRTOS ports que soportan interrupt nesting): */
void vAnInterruptServiceRoutine(void) {
    UBaseType_t uxSavedInterruptStatus;
    uxSavedInterruptStatus = taskENTER_CRITICAL_FROM_ISR();
    {
        /* Código protegido dentro de la ISR */
    }
    taskEXIT_CRITICAL_FROM_ISR(uxSavedInterruptStatus);
}
```

### 8.2.2 Suspensión del Scheduler

Para secciones críticas más largas (donde deshabilitar interrupciones sería demasiado), se puede suspender el scheduler. Las interrupciones permanecen habilitadas, pero no pueden ocurrir cambios de contexto.

```c
void vPrintString(const char *pcString) {
    /* Suspende el scheduler en lugar de deshabilitar interrupciones.
       Las interrupciones siguen activas, pero no habrá context switch. */
    vTaskSuspendAll();
    {
        printf("%s", pcString);
        fflush(stdout);
    }
    xTaskResumeAll();  /* Retorna pdTRUE si se realizó un context switch pendiente */
}
```

**`vTaskSuspendAll()` / `xTaskResumeAll()`:**
- Son anidables (el scheduler se reanuda cuando el contador llega a cero).
- No se deben llamar APIs de FreeRTOS mientras el scheduler está suspendido.
- `xTaskResumeAll()` retorna `pdTRUE` si se realizó un context switch que estaba pendiente.


## 8.3 Tipos de Semáforos en FreeRTOS

FreeRTOS ofrece tres tipos principales de semáforos, cada uno con un propósito distinto:

| Tipo | Creación | Uso principal | Valor inicial |
|------|----------|--------------|---------------|
| **Binario** | `xSemaphoreCreateBinary()` | Sincronización tarea↔ISR | 0 (vacío) |
| **Contador** | `xSemaphoreCreateCounting()` | Contar eventos, gestionar N recursos | configurable |
| **Mutex** | `xSemaphoreCreateMutex()` | Exclusión mutua entre tareas | 1 (disponible) |
| **Mutex Recursivo** | `xSemaphoreCreateRecursiveMutex()` | Exclusión mutua reentrante | 1 (disponible) |

**API común (salvo mutex recursivo):**

```c
/* Tomar (wait / acquire): bloquea hasta que el semáforo está disponible */
xSemaphoreTake(xSemaphore, xTicksToWait);

/* Dar (signal / release): libera el semáforo */
xSemaphoreGive(xSemaphore);

/* Dar desde ISR: */
xSemaphoreGiveFromISR(xSemaphore, &xHigherPriorityTaskWoken);
```

**Diferencia semáforo binario vs. mutex:**
- El **semáforo binario** se usa para sincronización: se da y normalmente no se devuelve (la ISR da, la tarea toma).
- El **mutex** se usa para exclusión mutua: siempre debe devolverse. Incluye mecanismo de **herencia de prioridad** que el semáforo binario no tiene.


### 8.3.1 Semáforo Binario — Sincronización Tarea↔ISR

El semáforo binario actúa como token de un solo lugar: puede estar "dado" (1) o "tomado" (0). Es ideal para que una ISR despierte una tarea que procesa el evento.

```c
#include "FreeRTOS.h"
#include "semphr.h"

static SemaphoreHandle_t xSemaphore = NULL;

/* ISR: da el semáforo para despertar la tarea */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    BaseType_t xHigherPriorityTaskWoken = pdFALSE;

    xSemaphoreGiveFromISR(xSemaphore, &xHigherPriorityTaskWoken);

    /* Si la tarea despertada tiene mayor prioridad que la tarea interrumpida,
       solicitar un context switch inmediato al salir de la ISR */
    portYIELD_FROM_ISR(xHigherPriorityTaskWoken);
}

/* Tarea: espera bloqueada hasta recibir el semáforo */
static void vHandlerTask(void *pvParameters) {
    for (;;) {
        /* Bloquea indefinidamente hasta que la ISR dé el semáforo */
        xSemaphoreTake(xSemaphore, portMAX_DELAY);
        /* Procesa el evento — solo llega aquí cuando el semáforo fue dado */
        vProcessEvent();
    }
}

int main(void) {
    xSemaphore = xSemaphoreCreateBinary();
    xTaskCreate(vHandlerTask, "Handler", 128, NULL, 2, NULL);
    vTaskStartScheduler();
    for (;;);
}
```

**Flujo de ejecución:**
1. La tarea llama `xSemaphoreTake()` → entra en estado **Blocked** (0% CPU).
2. Ocurre el evento → ISR ejecuta `xSemaphoreGiveFromISR()`.
3. La tarea sale del estado Blocked → pasa a Ready → Running.
4. La tarea procesa el evento y vuelve a `xSemaphoreTake()` → Blocked de nuevo.


### 8.3.2 Semáforo Contador

El semáforo contador mantiene un conteo entero. Útil para:
1. **Contar eventos:** cada `Give` incrementa, cada `Take` decrementa. El conteo refleja eventos pendientes de procesar.
2. **Gestionar N recursos:** valor inicial = N disponibles.

```c
/* Creación: valor máximo = 10, valor inicial = 0 */
SemaphoreHandle_t xCountingSemaphore = xSemaphoreCreateCounting(10, 0);

/* ISR: por cada evento, da el semáforo (incrementa el conteo) */
void vExampleISR(void) {
    BaseType_t xHigherPriorityTaskWoken = pdFALSE;
    xSemaphoreGiveFromISR(xCountingSemaphore, &xHigherPriorityTaskWoken);
    portYIELD_FROM_ISR(xHigherPriorityTaskWoken);
}

/* Tarea: procesa un evento por iteración */
static void vHandlerTask(void *pvParameters) {
    for (;;) {
        /* xSemaphoreTake retorna inmediatamente si el conteo > 0,
           decrementa el conteo en 1 cada vez */
        xSemaphoreTake(xCountingSemaphore, portMAX_DELAY);
        /* Procesa exactamente un evento */
        vProcessOneEvent();
    }
}
```

**Ventaja sobre semáforo binario:** si ocurren 3 eventos mientras la tarea está procesando, el conteo llega a 3. Al terminar, la tarea los procesa de a uno sin perder ninguno. El semáforo binario solo retendría 1 evento máximo.


## 8.4 Mutex — Exclusión Mutua

Un **Mutex** (MUTual EXclusion) es un semáforo binario especial diseñado para controlar acceso a recursos compartidos entre tareas. La palabra "token" describe su funcionamiento: solo la tarea que tiene el token puede acceder al recurso.

```c
SemaphoreHandle_t xSemaphoreCreateMutex(void);
```
- Retorna `NULL` si no hay heap suficiente.
- El mutex se crea en estado "disponible" (valor = 1, listo para ser tomado).

### Patrón de uso:

```c
/* Ejemplo: proteger acceso a stdout con un mutex */
static SemaphoreHandle_t xMutex;

static void prvNewPrintString(const char *pcString) {
    /* Tomar el mutex — bloquea hasta obtenerlo.
       Solo UNA tarea puede estar en esta sección a la vez. */
    xSemaphoreTake(xMutex, portMAX_DELAY);
    {
        printf("%s", pcString);
        fflush(stdout);
        /* El mutex DEBE ser devuelto siempre */
    }
    xSemaphoreGive(xMutex);
}

static void prvPrintTask(void *pvParameters) {
    char *pcStringToPrint = (char *)pvParameters;
    for (;;) {
        prvNewPrintString(pcStringToPrint);
        vTaskDelay(rand() % 0x20);
    }
}

int main(void) {
    xMutex = xSemaphoreCreateMutex();
    if (xMutex != NULL) {
        xTaskCreate(prvPrintTask, "Print1", 1000,
                    "Task 1 *********************\r\n", 1, NULL);
        xTaskCreate(prvPrintTask, "Print2", 1000,
                    "Task 2 ---------------------\r\n", 2, NULL);
        vTaskStartScheduler();
    }
    for (;;);
}
```

**Diferencia clave mutex vs semáforo binario:**
- El semáforo binario usado para sincronización generalmente **NO se devuelve** (se descarta tras procesar el evento).
- El mutex para exclusión mutua **SIEMPRE debe devolverse** — el mismo patrón take/give en toda operación.


## 8.5 Inversión de Prioridad (Priority Inversion)

La inversión de prioridad ocurre cuando una tarea de alta prioridad (HP) debe esperar a que una tarea de baja prioridad (LP) libere un mutex.

**Escenario problemático:**

```
Tiempo →
LP toma el mutex
                HP intenta tomar el mutex → bloqueada esperando LP
                         MP se activa y desaloja LP (pues HP está bloqueada)
                                   LP no puede ejecutar → HP sigue esperando
                                             MP termina → LP ejecuta → HP continúa
```

**El problema:** HP queda bloqueada esperando a LP, pero LP no puede ejecutar porque MP (prioridad media) la desaloja. HP termina esperando a MP también, aunque MP no tiene nada que ver con el mutex.

Esto se llama **inversión de prioridad no acotada** (*unbounded priority inversion*): la tarea de mayor prioridad podría esperar indefinidamente.

```
                 [LP toma mutex]
HP  ─────────────────────────────────── espera mutex ──────────── continúa
                                ↗                      ↘
MP  ────────────────────────── ejecuta (desaloja LP) ──── fin
LP  ────── [toma mutex] ─────── espera MP ─────────────── [da mutex]
```



## 8.6 Herencia de Prioridad (Priority Inheritance)

FreeRTOS implementa un mecanismo básico de herencia de prioridad en los **mutex** (no en semáforos binarios). Este mecanismo **reduce** el impacto de la inversión de prioridad haciéndola acotada (*bounded*).

**Funcionamiento:** Cuando HP se bloquea esperando un mutex que LP tiene, LP hereda temporalmente la prioridad de HP. Esto evita que MP desaloje a LP.

```
                 [LP toma mutex]
HP  ──────────────────────── espera mutex ──── continúa
                       ↗              ↘
MP  ─────────────────────────────────── espera (LP tiene prio HP, no puede ser desalojada)
LP  ── [toma mutex] ── hereda prio HP ── [da mutex, vuelve a prio original]
```

**Comportamientos específicos de la herencia de prioridad en FreeRTOS:**
- Una tarea puede heredar prioridad de múltiples mutexes que espera.
- La tarea mantiene la prioridad más alta heredada hasta liberar todos sus mutexes.
- La prioridad se restablece automáticamente al original al dar el mutex.
- **Por eso los mutexes NO pueden usarse desde ISRs:** afectarían la prioridad de tareas.

```c
/* La herencia es automática — no requiere código adicional.
   Solo usar xSemaphoreCreateMutex() en lugar de xSemaphoreCreateBinary() */

/* Tarea LP — obtiene el mutex antes de ser desalojada por HP */
static void vLowPriorityTask(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xMutex, portMAX_DELAY);
        /* Aquí LP hereda prio de HP si HP está esperando este mutex */
        vDoSlowOperation();
        xSemaphoreGive(xMutex);
        /* Al dar el mutex, la prioridad vuelve a la original de LP */
    }
}
```

> **Importante:** La herencia de prioridad **no elimina** la inversión de prioridad, solo la **limita** en el tiempo. No es buena práctica diseñar el sistema confiando en ella. El mejor diseño evita que tareas de diferente prioridad compartan mutexes.


## 8.7 Deadlock

El **deadlock** ocurre cuando dos (o más) tareas están bloqueadas esperando mutuamente recursos que la otra tiene. Ninguna puede avanzar.

**Escenario clásico con dos mutexes (X e Y):**

1. Tarea A toma mutex X exitosamente.
2. Tarea A es desalojada por Tarea B.
3. Tarea B toma mutex Y exitosamente.
4. Tarea B intenta tomar mutex X → **bloqueada** (lo tiene A).
5. Tarea A reanuda e intenta tomar mutex Y → **bloqueada** (lo tiene B).
6. **Deadlock:** A espera a B, B espera a A. Ninguna avanza.

```
  Tarea A              Tarea B
  ───────              ───────
  TAKE mutex X  ✓      TAKE mutex Y  ✓
  (desalojada)
                       TAKE mutex X  ✗  → Blocked
  TAKE mutex Y  ✗  → Blocked

  [A y B bloqueadas para siempre]
```

**Estrategias para evitar deadlock:**

1. **Timeout en lugar de `portMAX_DELAY`:** si no se obtiene el mutex en tiempo razonable, es síntoma de un error de diseño.

```c
if (xSemaphoreTake(xMutex, pdMS_TO_TICKS(100)) == pdPASS) {
    /* Operación protegida */
    xSemaphoreGive(xMutex);
} else {
    /* Timeout: manejar error — posible deadlock en el diseño */
    vHandleDeadlockError();
}
```

2. **Ordenamiento consistente:** si múltiples mutexes son necesarios, todas las tareas deben tomarlos en el mismo orden.

3. **Diseño que evita compartir:** la mejor solución es que cada recurso sea accedido por una sola tarea (patrón Gatekeeper).



## 8.8 Mutex Recursivo

Un mutex estándar causa deadlock si la misma tarea intenta tomarlo dos veces sin haberlo devuelto. El **mutex recursivo** permite que la misma tarea lo tome múltiples veces.

**Regla:** debe haber exactamente una llamada a `Give` por cada llamada a `Take`.

```c
/* API del mutex recursivo — prototipo idéntico pero funciones distintas: */
SemaphoreHandle_t xSemaphoreCreateRecursiveMutex(void);
xSemaphoreTakeRecursive(xRecursiveMutex, xTicksToWait);
xSemaphoreGiveRecursive(xRecursiveMutex);
```

```c
SemaphoreHandle_t xRecursiveMutex;

void vTaskFunction(void *pvParameters) {
    const TickType_t xMaxBlock20ms = pdMS_TO_TICKS(20);

    xRecursiveMutex = xSemaphoreCreateRecursiveMutex();

    for (;;) {
        /* Primer Take — recursive call count = 1 */
        if (xSemaphoreTakeRecursive(xRecursiveMutex, xMaxBlock20ms) == pdPASS) {

            /* Segunda toma del mismo mutex por la misma tarea.
               Con mutex estándar → deadlock.
               Con mutex recursivo → solo incrementa el call count a 2. */
            xSemaphoreTakeRecursive(xRecursiveMutex, xMaxBlock20ms);

            /* Primer Give — call count baja a 1, mutex AÚN no liberado */
            xSemaphoreGiveRecursive(xRecursiveMutex);

            /* Segundo Give — call count baja a 0, mutex liberado */
            xSemaphoreGiveRecursive(xRecursiveMutex);
        }
    }
}
```

**Caso de uso típico:** una función que adquiere un mutex llama a una subfunción que también necesita el mismo mutex.

```c
/* Función de biblioteca que usa el mutex internamente */
void vLibraryFunction(void) {
    xSemaphoreTakeRecursive(xRecursiveMutex, portMAX_DELAY);
    /* Operación interna protegida */
    xSemaphoreGiveRecursive(xRecursiveMutex);
}

void vTaskUsingLibrary(void *pvParameters) {
    for (;;) {
        xSemaphoreTakeRecursive(xRecursiveMutex, portMAX_DELAY);  /* count = 1 */
        {
            vLibraryFunction();   /* count = 2 internamente, luego vuelve a 1 */
            /* Otras operaciones protegidas */
        }
        xSemaphoreGiveRecursive(xRecursiveMutex);  /* count = 0, mutex liberado */
    }
}
```


## 8.9 Mutexes y el Scheduler

### Caso 1: Tareas con diferente prioridad

Cuando LP libera el mutex que HP estaba esperando, HP preempta inmediatamente a LP.

```
t1: LP toma mutex
t2: HP intenta tomar mutex → Blocked
t3: LP da el mutex → HP desbloquea y preempta a LP inmediatamente
t4: HP ejecuta con el mutex
t5: HP da el mutex → HP continúa o LP retoma según prioridades
```

### Caso 2: Tareas con la misma prioridad (comportamiento no intuitivo)

Cuando Task2 da el mutex, Task1 pasa a Ready pero **Task2 no es desalojada inmediatamente**. Task1 debe esperar al siguiente tick interrupt.

```
t1: Task2 ejecuta su time slice, toma el mutex
t2: Task1 inicia su time slice, intenta tomar mutex → Blocked
t3: Task2 continúa (Task1 bloqueada, es el único ready)
t4: Task2 da el mutex → Task1 pasa a Ready, pero Task2 sigue en Running
t5: Task2 toma el mutex OTRA VEZ antes de que Task1 pueda ejecutar
→ Task1 no consigue CPU — starvation potencial
```

**Solución para tareas de igual prioridad usando mutex en loop cerrado:**

```c
void vFunction(void *pvParameter) {
    TickType_t xTimeAtWhichMutexWasTaken;
    for (;;) {
        xSemaphoreTake(xMutex, portMAX_DELAY);
        xTimeAtWhichMutexWasTaken = xTaskGetTickCount();

        vCopyTextToFrameBuffer(cTextBuffer);   /* Operación lenta */

        xSemaphoreGive(xMutex);

        /* Solo ceder la CPU si el tick count cambió mientras se tenía el mutex.
           Asegura que la otra tarea tenga oportunidad de ejecutar,
           sin desperdiciar tiempo si el mutex se liberó rápidamente. */
        if (xTaskGetTickCount() != xTimeAtWhichMutexWasTaken) {
            taskYIELD();
        }
    }
}
```


## 8.10 Tareas Gatekeeper (Portero)

Las tareas gatekeeper ofrecen **exclusión mutua sin riesgo de inversión de prioridad ni deadlock**. Es el patrón de diseño más limpio para acceso exclusivo a recursos.

**Principio:** Solo la tarea gatekeeper accede al recurso directamente. Cualquier otra tarea que necesite el recurso envía una petición a la cola del gatekeeper.

```
Tarea A ──→ [Cola] ──→ Gatekeeper ──→ Recurso (stdout, LCD, SPI, etc.)
Tarea B ──→ [Cola] ──/
   ISR ──→ [Cola] ──/  (xQueueSendToFrontFromISR)
```

```c
static QueueHandle_t xPrintQueue;

/* Gatekeeper: única tarea con acceso directo a stdout */
static void prvStdioGatekeeperTask(void *pvParameters) {
    char *pcMessageToPrint;
    for (;;) {
        xQueueReceive(xPrintQueue, &pcMessageToPrint, portMAX_DELAY);
        printf("%s", pcMessageToPrint);
        fflush(stdout);
    }
}

/* Tarea normal: envía puntero a la cola (no toca stdout directamente) */
static void prvPrintTask(void *pvParameters) {
    int iIndexToString = (int)pvParameters;
    const TickType_t xMaxBlockTimeTicks = 0x20;
    for (;;) {
        xQueueSendToBack(xPrintQueue, &(pcStringsToPrint[iIndexToString]), 0);
        vTaskDelay(rand() % xMaxBlockTimeTicks);
    }
}

/* Tick hook (contexto ISR): usa la versión interrupt-safe */
void vApplicationTickHook(void) {
    static int iCount = 0;
    if (++iCount >= 200) {
        xQueueSendToFrontFromISR(xPrintQueue, &(pcStringsToPrint[2]), NULL);
        iCount = 0;
    }
}

int main(void) {
    xPrintQueue = xQueueCreate(5, sizeof(char *));
    if (xPrintQueue != NULL) {
        xTaskCreate(prvPrintTask, "Print1", 1000, (void *)0, 1, NULL);
        xTaskCreate(prvPrintTask, "Print2", 1000, (void *)1, 2, NULL);
        /* Gatekeeper con prioridad baja: procesa cuando las print tasks están blocked */
        xTaskCreate(prvStdioGatekeeperTask, "Gatekeeper", 1000, NULL, 0, NULL);
        vTaskStartScheduler();
    }
    for (;;);
}
```

**Ventajas del patrón Gatekeeper:**
- **Sin inversión de prioridad:** no hay mutex que heredar prioridades.
- **Sin deadlock:** ninguna tarea espera a otra que a su vez espera a ella.
- **ISR-friendly:** la ISR puede usar `xQueueSendToFrontFromISR()` sin problemas.
- **Serialización natural:** la cola ordena los accesos FIFO (o LIFO si se usa `SendToFront`).

**Tradeoff de prioridad del gatekeeper:**
- Prioridad **baja** → mensajes se procesan cuando las demás tasks están bloqueadas (mayor latencia).
- Prioridad **alta** → mensajes se procesan inmediatamente, pero el gatekeeper preempta a tareas de menor prioridad.


## 8.11 Resumen Comparativo — Métodos de Exclusión Mutua

| Método | Deshabilita interrupciones | Bloquea tareas | Inversión de prioridad | Deadlock posible | Uso desde ISR |
|--------|:------------------------:|:--------------:|:---------------------:|:---------------:|:-------------:|
| Sección crítica (`taskENTER_CRITICAL`) | Sí | No | No aplica | No | No (usar `_FROM_ISR`) |
| Suspender scheduler | No | No | No aplica | No | No |
| Semáforo binario | No | Sí | Sí (sin herencia) | Posible | Solo `Give` |
| Mutex | No | Sí | Minimizada (herencia) | Posible | No |
| Mutex recursivo | No | Sí | Minimizada (herencia) | Reducido | No |
| Gatekeeper task | No | Indirecto (cola) | No | No | Sí (`FromISR`) |

### Guía de selección:

1. **Región muy corta, necesita proteger de ISRs** → Sección crítica básica.
2. **Región más larga, ISRs pueden seguir activas** → Suspender scheduler.
3. **Sincronización ISR → tarea (un evento a la vez)** → Semáforo binario.
4. **Contar N eventos o gestionar N recursos** → Semáforo contador.
5. **Exclusión mutua entre tareas** → Mutex (con herencia de prioridad).
6. **Función recursiva o biblioteca que requiere el mismo mutex** → Mutex recursivo.
7. **Recurso compartido con múltiples productores + máxima limpieza** → Tarea Gatekeeper.

### Configuración en `FreeRTOSConfig.h`:

```c
#define configUSE_MUTEXES              1  /* Habilita mutexes */
#define configUSE_RECURSIVE_MUTEXES    1  /* Habilita mutexes recursivos */
#define configUSE_COUNTING_SEMAPHORES  1  /* Habilita semáforos contadores */
```
